# Log Pruned-Coronal-Only Model to MLflow

Standalone recovery notebook. The Run 8a training completed and saved the model
to DBFS, but `mlflow.transformers.log_model()` failed.

**Root cause:** `UperNetForSemanticSegmentation` is NOT compatible with
`mlflow.transformers` flavor — it's not a Pipeline and not in the AutoModel
registry. Neither passing the model object nor the path works.

**Fix:** Use `mlflow.log_artifacts()` to log the raw model files. The model
can be loaded later with `UperNetForSemanticSegmentation.from_pretrained(path)`.

In [0]:
import os
import json
import mlflow

# ---------- Config ----------
EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"
FINAL_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/pruned-coronal-only"
METADATA_PATH = os.path.join(FINAL_MODEL_DIR, "ablation_metadata.json")

# ---------- Verify files exist ----------
assert os.path.isdir(FINAL_MODEL_DIR), f"Model dir not found: {FINAL_MODEL_DIR}"
assert os.path.isfile(os.path.join(FINAL_MODEL_DIR, "config.json")), "config.json missing"
print(f"Model directory: {FINAL_MODEL_DIR}")
print(f"Contents: {os.listdir(FINAL_MODEL_DIR)}")

metadata = None
if os.path.isfile(METADATA_PATH):
    with open(METADATA_PATH) as f:
        metadata = json.load(f)
    print(f"Metadata loaded: {metadata.get('ablation_tag')}, {metadata.get('num_labels')} classes")
else:
    print("WARNING: ablation_metadata.json not found")

# ---------- MLflow setup ----------
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- End any stale run ----------
try:
    mlflow.end_run()
except Exception:
    pass

# ---------- Log model artifacts ----------
# UperNet is NOT compatible with mlflow.transformers flavor (not a Pipeline,
# not in AutoModel registry). Use mlflow.log_artifacts() for raw file logging.
# Load later with: UperNetForSemanticSegmentation.from_pretrained(path)
with mlflow.start_run(run_name="pruned-coronal-only-673class-artifact-recovery"):
    mlflow.log_artifacts(FINAL_MODEL_DIR, artifact_path="model")
    print("Model artifacts logged to MLflow under 'model/'")

    if metadata:
        mlflow.log_artifact(METADATA_PATH, artifact_path="metadata")
        print("Metadata artifact logged")

        cross_axis = metadata.get("cross_axis_eval", {})
        for axis_name, vals in cross_axis.items():
            mlflow.log_metrics({
                f"diag_{axis_name}_miou": vals["mean_iou"],
                f"diag_{axis_name}_accuracy": vals["accuracy"],
            })
        if cross_axis:
            print(f"Cross-axis metrics logged: {list(cross_axis.keys())}")

    mlflow.log_params({
        "recovery_notebook": True,
        "source_dir": FINAL_MODEL_DIR,
        "num_labels": metadata.get("num_labels", "unknown") if metadata else "unknown",
        "ablation_tag": metadata.get("ablation_tag", "unknown") if metadata else "unknown",
    })

print("\nDone. MLflow run closed.")

Model directory: /dbfs/FileStore/allen_brain_data/models/pruned-coronal-only
Contents: ['ablation_metadata.json', 'config.json', 'model.safetensors', 'preprocessor_config.json', 'training_args.bin']
Metadata loaded: coronal-only, 673 classes
MLflow experiment: /Users/noel.nosse@grainger.com/histology-brain-segmentation (ID: 1345391216675532)


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

Uploading /dbfs/FileStore/allen_brain_data/models/pruned-coronal-only/model.safetensors:   0%|          | 0.00…

Model artifacts logged to MLflow under 'model/'
Metadata artifact logged
Cross-axis metrics logged: ['coronal', 'axial', 'sagittal']

Done. MLflow run closed.
